# Lektion 08 - Multi-Agent-Designmuster


## Einrichtung


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Warum Multi-Agenten-Systeme?

Aufgaben aus der realen Welt wie die Reiseplanung erfordern viele unterschiedliche Fachkenntnisse — Logistik, Ortskenntnisse, Budgetierung und mehr. Ein einzelner Agent, der versucht, alles zu bewältigen, wird schnell unübersichtlich.

Multi-Agenten-Systeme lösen dies durch **Spezialisierung**: Jeder Agent konzentriert sich auf ein Fachgebiet und liefert dadurch qualitativ hochwertigere Ergebnisse als ein Generalist. Sie verbessern auch die **Skalierbarkeit** — Sie können neue Agenten (z. B. einen Flugexperten, einen Restaurantkritiker) hinzufügen, ohne den bestehenden Arbeitsablauf neu schreiben zu müssen. Die Agenten arbeiten zusammen durch eine strukturierte Pipeline, die Kontext von einem zum nächsten weitergibt.


## Spezialisierte Agenten erstellen


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Aufbau eines sequenziellen Workflows

Mit `WorkflowBuilder` können Sie Agenten zu einem gerichteten Graphen verbinden. Hier erstellen wir eine einfache zweistufige Pipeline: Der **TravelPlanner** erstellt den Reiseplan, danach überprüft und verbessert der **TravelConcierge** ihn.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Weitere Agenten zum Workflow hinzufügen

Einer der größten Vorteile des Multi-Agenten-Musters ist, wie einfach es erweiterbar ist. Unten fügen wir einen **BudgetReviewer**-Agenten hinzu, der den Plan mit dem Budget des Reisenden abgleicht, Positionen kennzeichnet, die die Kosten möglicherweise über das Limit treiben, und kostensparende Alternativen vorschlägt. Der Workflow führt nun drei Agenten nacheinander aus:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Zusammenfassung

In dieser Lektion haben Sie gelernt:

1. **Spezialisierte Agenten erstellen** — jeder mit einer fokussierten Rolle (Planung, Concierge, Budgetüberprüfung).
2. **Agenten zu einem sequentiellen Workflow verbinden** mit `WorkflowBuilder` und `add_edge`.
3. **Ausgabe aus einer Multi-Agenten-Pipeline streamen**, dabei verfolgen, welcher Agent gerade spricht.
4. **Einen Workflow erweitern**, indem neue Agenten zur Kette hinzugefügt werden, ohne bestehende zu verändern.

Das Multi-Agenten-Design-Muster hält jeden Agenten einfach, während es reichhaltigere, gründlicher überprüfte Ergebnisse liefert, als es jeder einzelne Agent allein erreichen könnte.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, beachten Sie bitte, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner Ursprungssprache gilt als maßgebliche Quelle. Bei kritischen Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Verwendung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
